# ROGII Safe Typewell Beam Baseline

A leakage-safe typewell GR tracker. It starts from the last observed TVT_input, tracks a bounded TVT path against the typewell GR signature, and shrinks the correction toward the last-value baseline when the signature is ambiguous.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# Kaggle's mounted directory can differ between notebook runtimes.
sample_paths = sorted(Path('/kaggle/input').rglob('sample_submission.csv'))
if not sample_paths:
    raise FileNotFoundError('sample_submission.csv was not mounted')
DATA = sample_paths[0].parent
TEST = DATA / 'test'

def tw_arrays(df):
    x = df[['TVT','GR']].dropna().sort_values('TVT')
    tvt = x.TVT.to_numpy(float); gr = x.GR.to_numpy(float)
    tvt, order = tvt[np.argsort(tvt)], np.argsort(tvt)
    gr = gr[order]
    u, first = np.unique(tvt, return_index=True)
    if len(u) != len(tvt):
        gr = np.array([np.median(gr[tvt == v]) for v in u])
        tvt = u
    return tvt, gr

def nearest(values, value):
    j = int(np.searchsorted(values, value))
    if j <= 0: return 0
    if j >= len(values): return len(values)-1
    return j-1 if abs(values[j-1]-value) <= abs(values[j]-value) else j

def decode(hw, tw):
    known = hw[hw.TVT_input.notna()]
    unknown = hw[hw.TVT_input.isna()]
    if len(known) == 0 or len(unknown) == 0:
        return np.array([], dtype=float)
    tvt, tgr = tw_arrays(tw)
    if len(tvt) < 3:
        return np.full(len(unknown), float(known.TVT_input.iloc[-1]))
    last = float(known.TVT_input.iloc[-1])
    kgr = known[known.GR.notna()]
    if len(kgr) >= 20:
        residual = kgr.GR.to_numpy(float) - np.interp(kgr.TVT_input.to_numpy(float), tvt, tgr)
        sigma = float(np.clip(np.nanstd(residual), 10., 60.))
    else:
        sigma = 30.
    obs = unknown.GR.astype(float).interpolate(limit_direction='both').fillna(float(np.nanmedian(tgr)))
    obs = obs.rolling(5, center=True, min_periods=1).median().to_numpy(float)
    step = max(abs(float(np.nanmedian(np.diff(tvt)))), 1e-3)
    prev_idx = np.array([nearest(tvt, last)], dtype=int)
    prev_cost = np.array([0.], dtype=float)
    history_idx = []; history_parent = []
    offsets = np.arange(-3, 4, dtype=int)
    for value in obs:
        ci = np.clip(prev_idx[:,None] + offsets[None,:], 0, len(tvt)-1).ravel()
        pa = np.repeat(np.arange(len(prev_idx)), len(offsets))
        cost = prev_cost[pa] + ((float(value)-tgr[ci])/sigma)**2
        cost += .75 * np.abs(ci-prev_idx[pa])
        order = np.argsort(cost, kind='stable')
        chosen=[]; parents=[]; costs=[]
        for oi in order:
            c=int(ci[oi])
            if c in chosen: continue
            chosen.append(c); parents.append(int(pa[oi])); costs.append(float(cost[oi]))
            if len(chosen) >= 8: break
        history_idx.append(np.asarray(chosen, dtype=int))
        history_parent.append(np.asarray(parents, dtype=int))
        prev_idx=np.asarray(chosen, dtype=int); prev_cost=np.asarray(costs, dtype=float)
    path=np.empty(len(history_idx), dtype=float); parent=int(np.argmin(prev_cost))
    for r in range(len(history_idx)-1,-1,-1):
        path[r]=tvt[history_idx[r][parent]]
        parent=int(history_parent[r][parent])
    # Community-derived decoder, conservatively blended with last-value.
    path += last-path[0]
    return last + .20*np.clip(path-last, -60., 60.)

values = {}
for hw_path in sorted(TEST.glob('*__horizontal_well.csv')):
    wid = hw_path.name.split('__', 1)[0]
    tw_path = TEST / f'{wid}__typewell.csv'
    if not tw_path.exists():
        continue
    hw = pd.read_csv(hw_path); tw = pd.read_csv(tw_path)
    mask = hw.TVT_input.isna()
    pred = decode(hw, tw)
    for row_idx, value in zip(hw.index[mask], pred):
        values[f'{wid}_{row_idx}'] = float(value)

submission = pd.read_csv(sample_paths[0])[['id']].copy()
fallback = float(np.nanmedian(list(values.values()))) if values else 0.
submission['tvt'] = submission.id.map(values).fillna(fallback)
assert len(submission) == len(pd.read_csv(sample_paths[0]))
assert np.isfinite(submission.tvt).all()
submission.to_csv('/kaggle/working/submission.csv', index=False)
print('wrote', len(submission), 'rows; TVT range', submission.tvt.min(), submission.tvt.max())